# Module 2: Slot Analysis

## Core Objective
To analyze the genomic regions *not* defined as anchors—the "slot space"—in order to map, classify, and quantify the dynamics of the accessory genome.

## Key Workflow & Features

1.  **Slot Space Partitioning**:
    *   For each high-quality genome, precisely maps all non-anchor regions.
    *   Assigns each slot a `high-resolution` ID (for specific context) and a `low-resolution` ID (for aggregated analysis), reducing analytical noise.
    *   Characterizes each slot by its size in both genes and base pairs.

2.  **Accessory Module (AM) Discovery**:
    *   Implements a robust, dual-validation workflow to define functional accessory units:
        1.  **Statistical Co-occurrence**: Builds a network of accessory genes based on significant co-occurrence statistics (Phi coefficient, p-value).
        2.  **Physical Adjacency Validation**: Validates co-occurring gene communities only if their members are physically adjacent on the chromosome in a high proportion of genomes.
    *   Genes not part of a multi-gene module are defined as `single-gene` modules, ensuring all accessory content is accounted for.

3.  **Modular Dynamics Analysis**:
    *   Maps the location of every Accessory Module across all slots in the entire cohort. A multi-gene module is only considered present if *all* its members are found in a single slot.
    *   Calculates the **SPI (Structural Plasticity Index)** for each module: the total number of unique slot types it has been observed in.
    *   Classifies each module's structural behavior (e.g., `Structurally Conserved`, `Niche-Specific`, `Multi-Position`) based on its SPI and population frequency.

4.  **"Grey Zone" Gene Analysis**:
    *   Specifically tracks the distribution of `Soft Core` and `Orphan Core` (non-anchor strict core) genes to identify any preferential positioning within the slot space.

In [ ]:
touch slot_analysis/__init__.py

In [ ]:
# /GenoSynax_test/genosyntax/slot_analysis/__init__.py

import os
import json

from genosyntax.utils.file_ops import load_json, save_json
from genosyntax.slot_analysis.slot_partitioner import partition_slot_space, analyze_grey_zone_genes
from genosyntax.slot_analysis.module_discoverer import discover_multi_gene_modules, define_accessory_modules
from genosyntax.slot_analysis.dynamics_analyzer import analyze_module_dynamics

def run(args):
    """
    Entrypoint function for the 'analyze_slots' module.
    MODIFIED: Now includes a check to ensure that anchors were actually found
    in the previous step before proceeding.
    """
    if args.dry_run:
        return True

    output_dir = args.output_dir
    intermediate_dir = os.path.join(output_dir, "intermediate")
    reports_dir = os.path.join(output_dir, "reports")
    os.makedirs(intermediate_dir, exist_ok=True)
    os.makedirs(reports_dir, exist_ok=True)

    print("====== GenoSyntax: Running Module 'analyze_slots' ======")

    # Step 0: Load data
    architecture_def = load_json(args.architecture_file)
    if not architecture_def:
        print(f"[FATAL] Could not load architecture definition file from: {args.architecture_file}")
        return False # Indicate failure for pipeline

    # Check if the previous step actually found any anchors.
    if not architecture_def.get("anchor_definitions"):
        print("\n[INFO] The architecture definition file contains no anchor definitions.")
        print("This indicates that the 'define_architecture' step did not find a stable backbone.")
        print("[ACTION] Skipping 'analyze_slots' module as it cannot run without anchors.")
        print(f"\n====== 'analyze_slots' Module Skipped. ======")
        return True # Return True because this is an expected outcome, not a crash.
        
    gene_order_db = load_json(os.path.join(args.intermediate_dir_prev, 'gene_order_database.json'))
    gene_coord_db = load_json(os.path.join(args.intermediate_dir_prev, 'gene_coordinate_database.json'))
    pangenome_strat = load_json(os.path.join(args.intermediate_dir_prev, 'pangenome_stratification.json'))
    gene_annotation_map = load_json(os.path.join(args.intermediate_dir_prev, 'gene_annotation_map.json'))

    if not all([gene_order_db, gene_coord_db, pangenome_strat, gene_annotation_map]):
        print(f"[FATAL] Could not load one or more database files from: {args.intermediate_dir_prev}")
        return False
        
    # Steps 1-5: The actual analysis logic
    slot_database = partition_slot_space(architecture_def, gene_order_db, gene_coord_db)
    df_orphan, df_soft = analyze_grey_zone_genes(slot_database, pangenome_strat, architecture_def['anchor_definitions'])
    passed_genomes = architecture_def['metadata']['qc_summary']['passed_genomes_list']
    multi_gene_modules = discover_multi_gene_modules(
        pangenome_strat, gene_order_db, gene_coord_db, passed_genomes,
        phi_threshold=args.phi_threshold,
        p_value_threshold=args.p_value_threshold,
        adjacency_distance=args.adjacency_distance,
        support_threshold=args.support_threshold
    )
    
    accessory_modules = define_accessory_modules(pangenome_strat, multi_gene_modules, gene_annotation_map)
    module_dynamics_df = analyze_module_dynamics(accessory_modules, slot_database, pangenome_strat, passed_genomes, spi_resolution=args.spi_resolution)

    # Save outputs
    save_json(slot_database, os.path.join(intermediate_dir, "slot_database.json"))
    save_json(accessory_modules, os.path.join(intermediate_dir, "accessory_module_definitions.json"))
    if not df_orphan.empty: df_orphan.to_csv(os.path.join(reports_dir, "orphan_core_gene_distribution.csv"), index=False)
    if not df_soft.empty: df_soft.to_csv(os.path.join(reports_dir, "soft_core_gene_distribution.csv"), index=False)
    module_dynamics_df.to_csv(os.path.join(reports_dir, "module_dynamics_report.csv"), index=False)

    print(f"\n====== 'analyze_slots' Module Finished Successfully! ======")
    print(f"[*] Final reports saved to: {reports_dir}")
    return True

slot_analysis/slot_partitioner.py

In [ ]:
# /GenoSynax_test/genosyntax/slot_analysis/slot_partitioner.py

import json
from collections import defaultdict, Counter
from tqdm import tqdm
import pandas as pd

def _find_anchor_location(skeleton, genome_order, all_core_genes, tolerance):
    """
    A robust helper function to find the precise start and end indices of an anchor
    skeleton within a genome's gene order.
    """
    # ... (This function is correct and remains unchanged) ...
    skel_len = len(skeleton)
    order_len = len(genome_order)
    try:
        first_gene_indices = [i for i, g in enumerate(genome_order) if g == skeleton[0]]
    except ValueError:
        return None, None
    for start_pos in first_gene_indices:
        skel_ptr, genome_ptr, acc_count = 1, start_pos + 1, 0
        found_indices = [start_pos]
        while skel_ptr < skel_len and genome_ptr < order_len:
            if genome_order[genome_ptr] == skeleton[skel_ptr]:
                found_indices.append(genome_ptr)
                skel_ptr += 1
                acc_count = 0
            elif genome_order[genome_ptr] not in all_core_genes:
                acc_count += 1
                if acc_count > tolerance: break
            else: break
            genome_ptr += 1
        if skel_ptr == skel_len:
            return min(found_indices), max(found_indices)
    return None, None


def partition_slot_space(architecture_def, gene_order_db, gene_coord_db):
    """
    Partitions the slot space based on the provided architecture definition.
    """
    # ... (This function is now correct and remains unchanged from the previous fix) ...
    print("\n--- Task 1: Partitioning Slot Space ---")
    
    slot_database = defaultdict(dict)
    anchor_defs = {a['anchor_id']: a for a in architecture_def['anchor_definitions']}
    
    try:
        pangenome_strat = architecture_def['metadata']['pangenome_stratification']
        all_core_genes = set(pangenome_strat.get('strict_core', []))
        params = architecture_def.get('metadata', {}).get('analysis_parameters', {})
        tolerance = params.get('tolerance', 2)
        qc_summary = architecture_def.get('metadata', {}).get('qc_summary', {})
        passed_genomes = qc_summary.get('passed_genomes_list', [])
    except KeyError as e:
        print(f"[FATAL] A required key was not found in the architecture_definition.json metadata: {e}")
        return {}
        
    if not passed_genomes:
        print("[ERROR] No high-quality genomes found in the architecture definition. Aborting.")
        return {}

    for genome_id in tqdm(passed_genomes, desc="Partitioning genomes"):
        gene_order = gene_order_db.get(genome_id)
        # In the provided code, coords were loaded but not used in a way that would cause an error.
        # However, for robustness, we should ensure both exist.
        coords = gene_coord_db.get(genome_id)
        if not gene_order or not coords: continue

        located_anchors = []
        for anchor_id, anchor_info in anchor_defs.items():
            start_idx, end_idx = _find_anchor_location(
                tuple(anchor_info['skeleton']), gene_order, all_core_genes, tolerance
            )
            if start_idx is not None:
                located_anchors.append({'id': anchor_id, 'start_idx': start_idx, 'end_idx': end_idx})
        
        if not located_anchors: continue
        located_anchors.sort(key=lambda x: x['start_idx'])
        
        def get_slot_size_bp(start_gene_idx, end_gene_idx):
            # This function requires the 'coords' list, hence the check above.
            if start_gene_idx >= len(coords) or end_gene_idx > len(coords): return 0
            if start_gene_idx >= end_gene_idx: return 0
            end_bp = coords[start_gene_idx-1]['end'] if start_gene_idx > 0 else 0
            start_bp = coords[end_gene_idx]['start'] if end_gene_idx < len(coords) else (coords[-1]['end'] if coords else 0)
            return abs(start_bp - end_bp)

        is_consistent = qc_summary.get('orientation_analysis', {}).get('is_consistently_oriented', False)
        
        if is_consistent:
            first_anchor = located_anchors[0]
            if first_anchor['start_idx'] > 0:
                slot_id_high = "Slot(Upstream)"
                slot_content = gene_order[0:first_anchor['start_idx']]
                slot_database[genome_id][slot_id_high] = {
                    'slot_id_high_res': slot_id_high, 'slot_id_low_res': 'Slot(Upstream)',
                    'slot_type': 'Upstream', 'content': slot_content,
                    'slot_size_genes': len(slot_content),
                    'slot_size_bp': get_slot_size_bp(0, first_anchor['start_idx'])
                }
            last_anchor = located_anchors[-1]
            if last_anchor['end_idx'] < len(gene_order) - 1:
                slot_id_high = "Slot(Downstream)"
                slot_content = gene_order[last_anchor['end_idx']+1:]
                slot_database[genome_id][slot_id_high] = {
                    'slot_id_high_res': slot_id_high, 'slot_id_low_res': 'Slot(Downstream)',
                    'slot_type': 'Downstream', 'content': slot_content,
                    'slot_size_genes': len(slot_content),
                    'slot_size_bp': get_slot_size_bp(last_anchor['end_idx']+1, len(gene_order))
                }
                
        for i in range(len(located_anchors) - 1):
            anchor1, anchor2 = located_anchors[i], located_anchors[i+1]
            start_idx, end_idx = anchor1['end_idx'] + 1, anchor2['start_idx']
            if start_idx >= end_idx: continue
            
            slot_id_high = f"Slot({anchor1['id']} -> {anchor2['id']})"
            slot_content = gene_order[start_idx:end_idx]
            slot_database[genome_id][slot_id_high] = {
                'slot_id_high_res': slot_id_high, 'slot_id_low_res': slot_id_high,
                'slot_type': 'Inter-Anchor', 'content': slot_content,
                'slot_size_genes': len(slot_content),
                'slot_size_bp': get_slot_size_bp(start_idx, end_idx)
            }
            
        for anchor in located_anchors:
            anchor_id = anchor['id']
            skeleton = set(anchor_defs[anchor_id]['skeleton'])
            
            skeleton_indices_in_span = [i for i in range(anchor['start_idx'], anchor['end_idx'] + 1) if gene_order[i] in skeleton]
            skeleton_indices_sorted = sorted(skeleton_indices_in_span)
            
            for i in range(len(skeleton_indices_sorted) - 1):
                start_idx, end_idx = skeleton_indices_sorted[i] + 1, skeleton_indices_sorted[i+1]
                if start_idx >= end_idx: continue
                
                gene1 = gene_order[skeleton_indices_sorted[i]]
                gene2 = gene_order[skeleton_indices_sorted[i+1]]
                
                slot_id_high = f"Slot({gene1} -> {gene2}, in: {anchor_id})"
                slot_id_low = f"Slot(in: {anchor_id})"
                slot_content = gene_order[start_idx:end_idx]
                
                slot_database[genome_id][slot_id_high] = {
                    'slot_id_high_res': slot_id_high, 'slot_id_low_res': slot_id_low,
                    'slot_type': 'Intra-Anchor', 'content': slot_content,
                    'slot_size_genes': len(slot_content),
                    'slot_size_bp': get_slot_size_bp(start_idx, end_idx)
                }
    
    print("Slot space partitioning complete.")
    return slot_database


def analyze_grey_zone_genes(slot_database, pangenome_strat, anchor_defs):
    """
    Analyzes the distribution of 'Soft Core' and 'Orphan Core' genes within the slot space.
    MODIFIED: Handles the edge case where no 'grey zone' genes are found.
    """
    print("\n--- Task 2: Analyzing 'Grey Zone' Gene Distribution ---")
    
    soft_core_genes = set(pangenome_strat.get('soft_core', []))
    all_anchor_core_genes = {gene for anchor in anchor_defs for gene in anchor.get('skeleton', [])}
    orphan_core_genes = set(pangenome_strat.get('strict_core', [])) - all_anchor_core_genes
    
    soft_core_dist = defaultdict(Counter)
    orphan_core_dist = defaultdict(Counter)
    
    for genome_id, slots in slot_database.items():
        for slot_id, slot_info in slots.items():
            for gene in slot_info.get('content', []):
                if gene in soft_core_genes:
                    soft_core_dist[gene][slot_info['slot_id_low_res']] += 1
                elif gene in orphan_core_genes:
                    orphan_core_dist[gene][slot_info['slot_id_low_res']] += 1
                    
    # --- THIS IS THE MODIFIED PART ---
    def format_distribution_report(dist_dict, gene_type):
        rows = []
        for gene, counter in dist_dict.items():
            for slot_id, count in counter.items():
                rows.append({
                    'gene_id': gene,
                    'gene_type': gene_type,
                    'found_in_slot_id': slot_id,
                    'observation_count': count
                })
        
        # If no rows were created, return an empty DataFrame WITH the correct columns defined.
        if not rows:
            return pd.DataFrame(columns=['gene_id', 'gene_type', 'found_in_slot_id', 'observation_count'])
        return pd.DataFrame(rows)
    # --- END OF MODIFICATION ---

    df_soft_core = format_distribution_report(soft_core_dist, 'Soft_Core')
    df_orphan_core = format_distribution_report(orphan_core_dist, 'Orphan_Core')
    
    print(f"Found {df_orphan_core['gene_id'].nunique()} Orphan Core and {df_soft_core['gene_id'].nunique()} Soft Core genes in slots.")
    return df_orphan_core, df_soft_core

slot_analysis/module_discoverer.py

In [ ]:
import pandas as pd
from itertools import combinations
from scipy.stats import fisher_exact
import networkx as nx
import community as community_louvain
from tqdm import tqdm
import numpy as np
from collections import defaultdict

def _calculate_phi_coefficient(contingency_table):
    """Calculates Phi coefficient from a 2x2 contingency table."""
    n11, n10, n01, n00 = contingency_table.ravel()
    denominator = (n11 + n10) * (n01 + n00) * (n11 + n01) * (n10 + n00)
    return (n11 * n00 - n10 * n01) / np.sqrt(denominator) if denominator > 0 else 0

def discover_multi_gene_modules(
    pangenome_strat, 
    gene_order_db, 
    gene_coord_db, 
    passed_genomes, 
    phi_threshold=0.8, 
    p_value_threshold=0.01, 
    adjacency_distance=250, 
    support_threshold=0.8
):
    """
    Discovers multi-gene accessory modules (AMs) using a dual-validation workflow.
    1. Statistical co-occurrence network clustering.
    2. Physical adjacency validation.
    """
    print("\n--- Task 3: Discovering Multi-Gene Accessory Modules ---")
    
    accessory_genes = set(pangenome_strat['accessory'])
    num_genomes = len(passed_genomes)
    
    # --- 1a. Build gene presence matrix for accessory genes ---
    print("  Step 1: Calculating co-occurrence...")
    presence_matrix = defaultdict(list)
    for gene in accessory_genes:
        for genome in passed_genomes:
            presence_matrix[gene].append(1 if gene in gene_order_db.get(genome, []) else 0)
    
    df_presence = pd.DataFrame(presence_matrix, index=passed_genomes).T

    # --- 1b. Calculate significant co-occurring pairs ---
    significant_pairs = []
    for gene1, gene2 in tqdm(combinations(accessory_genes, 2), total=len(accessory_genes)*(len(accessory_genes)-1)//2, desc="  Finding pairs"):
        v1, v2 = df_presence.loc[gene1], df_presence.loc[gene2]
        n11, n10, n01, n00 = ((v1==1)&(v2==1)).sum(), ((v1==1)&(v2==0)).sum(), ((v1==0)&(v2==1)).sum(), ((v1==0)&(v2==0)).sum()
        
        contingency_table = [[n11, n10], [n01, n00]]
        phi = _calculate_phi_coefficient(np.array(contingency_table))
        
        if phi >= phi_threshold:
            _, p_val = fisher_exact(contingency_table, alternative='greater')
            if p_val < p_value_threshold:
                significant_pairs.append({'gene1': gene1, 'gene2': gene2, 'phi': phi, 'p_value': p_val})

    if not significant_pairs:
        print("  No significant co-occurring pairs found. No multi-gene modules discovered.")
        return []

    # --- 2. Community Detection ---
    print(f"\n  Step 2: Found {len(significant_pairs)} significant pairs. Detecting communities...")
    edges_df = pd.DataFrame(significant_pairs)
    G = nx.from_pandas_edgelist(edges_df, 'gene1', 'gene2', edge_attr='phi')
    partition = community_louvain.best_partition(G, weight='phi')
    
    candidate_modules = defaultdict(list)
    for gene, mod_id in partition.items():
        candidate_modules[mod_id].append(gene)
    
    # Filter out single-gene communities, as they will be handled later
    multi_gene_candidates = [members for members in candidate_modules.values() if len(members) > 1]
    print(f"  Found {len(multi_gene_candidates)} candidate multi-gene modules.")

    # --- 3. Physical Adjacency Validation ---
    print("\n  Step 3: Performing physical adjacency validation...")
    validated_modules = []
    for module_genes in tqdm(multi_gene_candidates, desc="  Validating modules"):
        module_genes_set = set(module_genes)
        genomes_to_check = [
            gid for gid in passed_genomes 
            if module_genes_set.issubset(set(gene_order_db.get(gid, [])))
        ]
        
        if not genomes_to_check:
            continue
            
        support_count = 0
        for genome_id in genomes_to_check:
            coords = gene_coord_db[genome_id]
            module_coords = [g for g in coords if g['group_id'] in module_genes_set]
            
            if len(module_coords) != len(module_genes): continue # Handle paralogs simply by size check

            module_coords.sort(key=lambda x: x['start'])
            
            # Check if all are on the same contig
            if len(set(g['contig'] for g in module_coords)) != 1:
                continue

            # Check adjacency
            is_adjacent = True
            for i in range(len(module_coords) - 1):
                gene_curr, gene_next = module_coords[i], module_coords[i+1]
                distance = gene_next['start'] - gene_curr['end']
                if distance < 0 or distance > adjacency_distance: # Genes overlap or are too far
                    is_adjacent = False
                    break
                
                # Check for intervening genes
                intervening_genes = [
                    g for g in coords 
                    if g['contig'] == gene_curr['contig'] and 
                       g['start'] > gene_curr['end'] and 
                       g['end'] < gene_next['start']
                ]
                if intervening_genes:
                    is_adjacent = False
                    break
            
            if is_adjacent:
                support_count += 1
        
        support_fraction = support_count / len(genomes_to_check) if genomes_to_check else 0
        if support_fraction >= support_threshold:
            validated_modules.append(sorted(module_genes))

    print(f"  Validation complete. Discovered {len(validated_modules)} high-confidence multi-gene modules.")
    return validated_modules


def define_accessory_modules(
    pangenome_strat, 
    multi_gene_modules,
    gene_annotation_map
):
    """
    Finalizes the Accessory Module (AM) list by combining multi-gene modules
    and defining all other accessory genes as single-gene modules.
    """
    print("\n--- Task 4: Finalizing Accessory Module Definitions ---")
    
    accessory_genes = set(pangenome_strat['accessory'])
    genes_in_multi_gene_modules = {gene for mod in multi_gene_modules for gene in mod}
    
    accessory_modules = []
    module_counter = 1
    
    # --- 1. Add validated multi-gene modules ---
    for members in multi_gene_modules:
        # Find a consensus function
        functions = [gene_annotation_map.get(g, 'hypothetical protein') for g in members]
        consensus_func = next((f for f in functions if 'hypothetical' not in f.lower()), 'hypothetical protein')
        
        accessory_modules.append({
            'module_id': f"AM_{module_counter:04d}",
            'module_type': 'multi-gene',
            'num_genes': len(members),
            'members': members,
            'consensus_function': consensus_func
        })
        module_counter += 1

    # --- 2. Add remaining accessory genes as single-gene modules ---
    single_genes = accessory_genes - genes_in_multi_gene_modules
    for gene in sorted(list(single_genes)):
        accessory_modules.append({
            'module_id': f"AM_{module_counter:04d}",
            'module_type': 'single-gene',
            'num_genes': 1,
            'members': [gene],
            'consensus_function': gene_annotation_map.get(gene, 'hypothetical protein')
        })
        module_counter += 1
        
    print(f"Defined a total of {len(accessory_modules)} accessory modules ({len(multi_gene_modules)} multi-gene, {len(single_genes)} single-gene).")
    return accessory_modules

slot_analysis/dynamics_analyzer.py

In [ ]:
import pandas as pd
from collections import defaultdict
from tqdm import tqdm

def analyze_module_dynamics(
    accessory_modules, 
    slot_database, 
    pangenome_strat,
    passed_genomes,
    spi_resolution='low'
):
    """
    Analyzes the structural dynamics of accessory modules (AMs).
    VERSION 2 (CORRECTED): Implements a strict definition for module presence in a slot.
    """
    print("\n--- Task 5: Analyzing Modular Dynamics (SPI Calculation - Corrected) ---")
    print(f"  Using SPI resolution: '{spi_resolution}'")

    module_location_map = defaultdict(set) # {module_id: set_of_observed_slot_keys}
    module_occurrence_map = defaultdict(set) # {module_id: set_of_genome_ids}
    
    # Create a reverse map from gene to module for quick lookup
    gene_to_module_map = {}
    for module in accessory_modules:
        for gene in module['members']:
            gene_to_module_map[gene] = module['module_id']

    # Pre-map modules to their member sets for efficient checking
    module_member_sets = {m['module_id']: set(m['members']) for m in accessory_modules}

    print("  Mapping module locations with strict presence definition...")
    for genome_id, slots in tqdm(slot_database.items(), desc="  Mapping module locations"):
        for slot_id, slot_info in slots.items():
            slot_content_set = set(slot_info['content'])
            slot_key = slot_info['slot_id_low_res'] if spi_resolution == 'low' else slot_info['slot_id_high_res']
            
            # --- STRICT MODULE PRESENCE CHECK ---
            # Find which modules have ALL their members inside this slot
            
            # Find all unique modules that have at least one member in the slot
            candidate_module_ids = {gene_to_module_map[g] for g in slot_content_set if g in gene_to_module_map}
            
            for module_id in candidate_module_ids:
                # A module is considered present in a slot ONLY IF
                # all of its members are found within that slot's content.
                if module_member_sets[module_id].issubset(slot_content_set):
                    module_location_map[module_id].add(slot_key)
                    module_occurrence_map[module_id].add(genome_id)

    print("\n  Calculating SPI and summarizing module dynamics...")
    results = []
    num_total_genomes = len(passed_genomes)

    for module in tqdm(accessory_modules, desc="  Analyzing modules"):
        module_id = module['module_id']
        
        # Get data from our new, correct maps
        observed_slots = module_location_map.get(module_id, set())
        num_genomes_with_module = len(module_occurrence_map.get(module_id, set()))
        
        frequency = num_genomes_with_module / num_total_genomes if num_total_genomes > 0 else 0
        spi = len(observed_slots)
        
        behavioral_class = "N/A"
        if spi > 1: behavioral_class = "Multi-Position"
        elif spi == 1:
            if frequency > 0.5: behavioral_class = "Structurally Conserved"
            else: behavioral_class = "Niche-Specific"
        elif spi == 0:
             behavioral_class = "Not Observed in Slots"

        results.append({
            'module_id': module['module_id'],
            'module_type': module['module_type'],
            'num_genes': module['num_genes'],
            'members': ", ".join(module['members']),
            'consensus_function': module['consensus_function'],
            'frequency': round(frequency, 4),
            'occurrence_count': num_genomes_with_module,
            'spi': spi,
            'behavioral_class': behavioral_class,
            'observed_slots': "; ".join(sorted(list(observed_slots))) if observed_slots else "None"
        })

    report_df = pd.DataFrame(results).sort_values(by=['spi', 'frequency'], ascending=[False, False])
    
    print("Modular dynamics analysis complete.")
    return report_df